In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
q12025 = pd.read_excel('data/2025_q1.xlsx', header=9)
q22025 = pd.read_excel('data/2025_q2.xlsx', header=9)
q32025 = pd.read_excel('data/2025_q3.xlsx', header=9)
q42025 = pd.read_excel('data/2025_q4.xlsx', header=9)

In [ ]:
q12025['TRIMESTRE'] = 1
q22025['TRIMESTRE'] = 2
q32025['TRIMESTRE'] = 3
q42025['TRIMESTRE'] = 4

In [ ]:
df2025total = pd.concat([q12025, q22025, q32025, q42025], ignore_index=True)

In [ ]:
totalcomparendos2025 = df2025total['CANTIDAD'].sum()
print(totalcomparendos2025)

In [ ]:
poblacion2025 = pd.read_excel('Poblacion2025TotalMunicipios.xlsx')
poblacion2025['CÓDIGO DIVIPOLA'] = pd.to_numeric(
    poblacion2025['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')

In [ ]:
df2025total = df2025total.merge(
    poblacion2025[['CÓDIGO DIVIPOLA', 'PoblacionTotal2025']],
    on='CÓDIGO DIVIPOLA',
    how='left'
)

In [ ]:
municipio2025 = df2025total.groupby(
    ['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'],
    as_index=False
).agg({
    'CANTIDAD': 'sum',
    'PoblacionTotal2025': 'max'
})

municipio2025.columns = ['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO',
                          'comparendos', 'poblacion']

municipio2025['tasacomparendos100k'] = (
    municipio2025['comparendos'] / municipio2025['poblacion'] * 100000)

In [ ]:
top10tasas = municipio2025.nlargest(10, 'tasacomparendos100k')
top10tasas[['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO',
            'comparendos', 'poblacion', 'tasacomparendos100k']]

In [ ]:
def clasificar_categoria(pob):
    if pob > 500001:
        return 'Categoría especial'
    elif 100001 <= pob <= 500000:
        return 'Categoría 1'
    elif 50001 <= pob <= 100000:
        return 'Categoría 2'
    elif 30001 <= pob <= 50000:
        return 'Categoría 3'
    elif 20001 <= pob <= 30000:
        return 'Categoría 4'
    elif 10001 <= pob <= 20000:
        return 'Categoría 5'
    else:
        return 'Categoría 6'

municipio2025['categoria'] = municipio2025['poblacion'].apply(
    clasificar_categoria)

In [ ]:
resumen_categorias = {}

for cat in municipio2025['categoria'].unique():
    df_cat = municipio2025[municipio2025['categoria'] == cat]
    resumen_categorias[cat] = df_cat
    df_cat.to_excel(f'comparendos_mpios_categoria_{cat}_2025.xlsx',
                    index=False)

In [ ]:
df2025total.to_csv('Comparendos2025conPoblacionyTasa.csv', index=False)